# PerfectPetClub - Data Engineering ETL Pipeline
## Customer Churn Prediction & Analytics

### 1. Project Overview & Business Problem
**PerfectPetClub** is an online subscription service providing pet food and supplies. A key business challenge is **customer churn**—when customers disable the auto-renewal feature of their subscription (`auto_renew = 0`).

This ETL pipeline extracts data across relational and transactional sources, validates data quality, engineers predictive features for churn analysis, and exports the final curated dataset into an optimized Apache Parquet format.

### 2. Architecture: Extract - Transform - Load (ETL)
* **Extract:** Ingest customer profiles, subscriptions, sales, payments, and product reviews from SQLite (`PerfectPetClub.db`).
* **Transform:**
  1. *Data Quality Validation:* Run automated assertion checks (null handling, date order logic, identifier uniqueness, boundary checks).
  2. *Feature Engineering:* Construct the binary churn target (`is_churned`), subscription tenure, financial aggregations, and customer feedback metrics.
  3. *Data Ethics & GDPR:* Mask/drop personally identifiable information (PII) such as passwords and street addresses.
* **Load:** Write the clean customer-level feature dataset to `output/perfectpetclub_churn_features.parquet`.

In [64]:
import sys
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import logging

sys.path.insert(0, str(Path("../src").resolve()))

%load_ext autoreload
%autoreload 2

from extract_customer_subscriptions import (
    extract_customer_subscriptions,
    validate_customer_data,
    transform_customer_features
)

print('Environment and modules loaded successfully!')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Environment and modules loaded successfully!



### Stage 1: Extract — Customer and Subscription Data

This stage extracts customer and subscription information from the SQLite database `PerfectPetClub.db`.

The extraction uses SQL INNER JOIN operations across three tables:

- **customer:** Customer ID, name, email, county and postcode.
- **subscription:** Subscription ID, start date, end date and auto-renewal status.
- **subscription_type:** Subscription tier and delivery service.

The extracted dataset contains 221 customer records, which are passed to the validation and feature engineering stages.


In [65]:
DB_PATH = '../data/PerfectPetClub.db'

raw_customers = extract_customer_subscriptions(db_path=DB_PATH)

print(f'Extracted {len(raw_customers)} customer records.')
raw_customers.head(3)

2026-11-11 16:51:24,819 - INFO - Connecting to database: ../data/PerfectPetClub.db
2026-11-11 16:51:24,848 - INFO - Customer count verified: 221 records.
2026-11-11 16:51:24,849 - INFO - Successfully extracted 221 customer records.


Extracted 221 customer records.


,customer_id,customer_name,email,county,postcode,subscription_id,sub_start_date,sub_end_date,auto_renew,subscription_tier,delivery_service
0,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,Berkshire,SL6 1CD,1,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,SuperSaver,Economy72
1,2,Leyla Mammadova,Mammadova282@example.com,Oxfordshire,OX7 5GH,2,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,SuperSaver,Economy72
2,3,Sana Shah,Sana858@notrealemail.com,Worcestershire,DY10 2JK,3,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24


## Stage 2: Transform — Automated Data Quality Checks
Before performing feature engineering, we validate data consistency, integrity, and domain rules:
1. **Uniqueness:** Verify all `customer_id` values are unique.
2. **Missing Values:** Ensure primary identifiers and renewal statuses have zero nulls.
3. **Date Logic:** Check that `sub_start_date` <= `sub_end_date`.
4. **Value Boundaries:** Check non-negative spend values and review ratings bounded within [0, 5].

In [66]:
print("--- Validating Customer Data ---")

cust_dq = validate_customer_data(raw_customers)

print("Customer validation results:")
for check, result in cust_dq.items():
    print(f"{check}: {result}")

2026-11-11 16:51:30,579 - INFO - Executing automated Data Quality checks on customer subscriptions...
2026-11-11 16:51:30,616 - INFO - All Customer Data Quality checks PASSED successfully (5/5).


--- Validating Customer Data ---
Customer validation results:
null_checks: {'customer_id': 0, 'email': 0, 'subscription_id': 0, 'auto_renew': 0}
is_unique: True
valid_auto_renew: True
valid_dates: True
date_logic_valid: True


In [67]:

# Test: Can our validation detect an invalid date?

test_data = raw_customers.copy()

# Deliberately insert an invalid subscription date
test_data.loc[0, 'sub_end_date'] = 'invalid-date'

try:
    validate_customer_data(test_data)
    print("TEST FAILED: Invalid date was accepted.")

except AssertionError as e:
    print("TEST PASSED: Invalid date was detected.")
    print(e)


2026-11-11 16:51:38,643 - INFO - Executing automated Data Quality checks on customer subscriptions...


TEST PASSED: Invalid date was detected.
DQ FAIL: Missing or invalid subscription dates.


/Users/roshangauchan/Documents/DataEngineering/DataEngineeringPresentation/src/extract_customer_subscriptions.py:101: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  


### Stage 3: Transform — Customer Feature Engineering

In this stage, we create features from the customer and subscription data to support churn analysis.

- **is_churned:** Binary target variable (1 if auto-renew is disabled, otherwise 0).
- **subscription_tenure_days:** Number of days between the recorded subscription start and end dates.
- **postcode_area:** Leading letters of the postcode used for geographical analysis.

The transformed dataset is checked before integration with other team members' data.

In [68]:
# 1. Transform your customer and subscription data
clean_customers = transform_customer_features(raw_customers)

# 2. Remove unnecessary identifiers
final_dataset = clean_customers.drop(
    columns=['customer_name', 'postcode', 'subscription_id']
)

# 3. Check your processed data
print("Total customers:", len(final_dataset))
print("Churned customers:", final_dataset['is_churned'].sum())
print("Dataset shape:", final_dataset.shape)

display(final_dataset.head())

2026-11-11 16:51:51,505 - INFO - Applying feature engineering on customer subscriptions...
2026-11-11 16:51:51,527 - INFO - Target Engineered: 86/221 customers churned (38.91%).


Total customers: 221
Churned customers: 86
Dataset shape: (221, 11)


,customer_id,email,county,sub_start_date,sub_end_date,auto_renew,subscription_tier,delivery_service,is_churned,subscription_tenure_days,postcode_district
0,1,Sargsyan701@somemail.co.uk,Berkshire,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,SuperSaver,Economy72,1,30,SL6
1,2,Mammadova282@example.com,Oxfordshire,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,SuperSaver,Economy72,0,330,OX7
2,3,Sana858@notrealemail.com,Worcestershire,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24,0,330,DY10
3,4,Omar.Farooq4@example.com,Kent,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24,0,330,TN9
4,5,Alejandro357@somemail.co.uk,Hertfordshire,2026-01-01 00:00:00.000,2026-05-01 00:00:00.000,0,SuperSaver,Economy72,1,120,WD7


# Churn rate by subscription tier

In [69]:
# Analyse churn rate by subscription tier

tier_summary = final_dataset.groupby("subscription_tier")["is_churned"].agg(
    total_customers="count",
    churn_rate="mean"
).reset_index()

# Convert churn rate into percentage
tier_summary["churn_rate"] = (
    tier_summary["churn_rate"] * 100
).round(2)

# Display results
print("--- Churn Rate by Subscription Tier ---")
display(tier_summary)

--- Churn Rate by Subscription Tier ---


,subscription_tier,total_customers,churn_rate
0,Basic,77,33.77
1,Premium,47,40.43
2,SuperSaver,97,42.27


## Stage 4: Load — Export Curated Dataset to Apache Parquet
As specified in Task 2, the final transformed dataset must be exported as an Apache Parquet file.
Parquet provides columnar compression, preserves column data types, and is optimized for downstream analytical queries.

In [ ]:
from pathlib import Path
import pandas as pd

# Create output folder
output_dir = Path("../output")
output_dir.mkdir(parents=True, exist_ok=True)

# Save only my customer and subscription features
parquet_path = output_dir / "customer_subscription_features.parquet"

# Export to Parquet
final_dataset.to_parquet(parquet_path, index=False)

# Read the saved file to verify
verified = pd.read_parquet(parquet_path)

print("Parquet saved successfully!")
print("Total customers:", len(verified))
print("Total columns:", len(verified.columns))
print("File location:", parquet_path.resolve())

# Verification: Read back parquet file to guarantee integrity

In [ ]:
# Verify the new customer-only Parquet file
df_verified = pd.read_parquet(parquet_path)

print("Parquet verification passed!")
print("Rows:", df_verified.shape[0])
print("Columns:", df_verified.shape[1])
print("Columns match:", list(df_verified.columns) == list(final_dataset.columns))

In [ ]:
# Final validation of customer subscription features

# 1. Check unique customer IDs
assert final_dataset["customer_id"].is_unique

# 2. Check customer count has not changed
assert len(final_dataset) == len(raw_customers)

# 3. Verify churn calculation
expected_churn = (
    final_dataset["auto_renew"] == 0
).astype(int)

assert (final_dataset["is_churned"] == expected_churn).all()

# 4. Check subscription durations
assert (final_dataset["subscription_tenure_days"] >= 0).all()

# 5. Verify exported Parquet structure
assert df_verified.shape == final_dataset.shape
assert list(df_verified.columns) == list(final_dataset.columns)

print("All 5 final validation checks PASSED!")